# Thermal cascade — train + test (YOLO26x)
Runtime → Change runtime type → **GPU** first.

**You do:** zip the two dataset folders and upload the zips to Google Drive:
```bash
cd /Volumes/dronisight
zip -r transformer.zip yolo_thermal_transformer
zip -r wire.zip Yolo_thermal_wire
```
**This notebook does:** mount Drive → unzip → train `transformer` + `wire` YOLO26x → evaluate → run the full cascade inference (this repo's code) on test images.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pins that matter on Colab: 8.3.x silently degrades yolo26 -> nano;
# pillow 11.3+ is broken. Then clone this repo for the cascade inference code.
!pip install -q -U "ultralytics>=8.4.60" "pillow==11.2.1"
!git clone -q https://github.com/arupa444/transformer_train.git /content/transformer_train
import sys; sys.path.insert(0, '/content/transformer_train/src')
print('ok')

### Point these at YOUR zips on Drive
Edit the two paths to wherever you uploaded `transformer.zip` / `wire.zip`.

In [ ]:
TRANSFORMER_ZIP = '/content/drive/MyDrive/transformer.zip'
WIRE_ZIP        = '/content/drive/MyDrive/wire.zip'
# where trained weights are copied so they survive the session
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/thermal_weights'

In [ ]:
import zipfile, os, yaml
for z in (TRANSFORMER_ZIP, WIRE_ZIP):
    with zipfile.ZipFile(z) as zf:
        zf.extractall('/content')
    print('unzipped', z)
DATASETS = {'transformer': '/content/yolo_thermal_transformer',
            'wire': '/content/Yolo_thermal_wire'}
# repoint each data.yaml `path:` (built on the Mac) to the Colab location
DATA_YAML = {}
for key, root in DATASETS.items():
    p = f'{root}/data_clahe.yaml'
    d = yaml.safe_load(open(p)); d['path'] = root
    yaml.safe_dump(d, open(p, 'w'), sort_keys=False)
    DATA_YAML[key] = p
    print(key, '->', d)

In [ ]:
# Thermal-tuned. NO hue jitter (palette = heat). x is heavy for ~600-700 imgs ->
# early stopping + regularization. On OOM: MODEL='yolo26m.pt' or lower batch.
MODEL = 'yolo26x.pt'
def train_args(data_yaml, scale):
    return dict(data=data_yaml, epochs=150, imgsz=1280, batch=4, seed=1337,
                hsv_h=0.0, hsv_s=0.2, hsv_v=0.3,
                fliplr=0.5, flipud=0.0, degrees=10.0, translate=0.1, scale=scale,
                mosaic=1.0, close_mosaic=10,
                weight_decay=0.0005, dropout=0.1, cos_lr=True, patience=30, amp=True)

In [ ]:
from ultralytics import YOLO
m_t = YOLO(MODEL)  # full frame, big object
m_t.train(project='runs/transformer', name='yolo26x',
          **train_args(DATA_YAML['transformer'], scale=0.5))

In [ ]:
m_w = YOLO(MODEL)  # transformer crops, thin objects -> wider scale jitter
m_w.train(project='runs/wire', name='yolo26x',
          **train_args(DATA_YAML['wire'], scale=0.9))

In [ ]:
# Persist weights to Drive (survive runtime resets) and into the repo's models/.
import os, shutil
os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)
os.makedirs('/content/transformer_train/models', exist_ok=True)
T = 'runs/transformer/yolo26x/weights/best.pt'
W = 'runs/wire/yolo26x/weights/best.pt'
for src, name in ((T, 'transformer.pt'), (W, 'wire.pt')):
    shutil.copy(src, f'{DRIVE_WEIGHTS_DIR}/{name}')
    shutil.copy(src, f'/content/transformer_train/models/{name}')
print('weights saved to', DRIVE_WEIGHTS_DIR, 'and repo models/')

## Test
Per-model mAP on the held-out **test** split, then the full **cascade** (transformer → crop → wire → relative-heat CV) on test images.

In [ ]:
for tag, m, key in (('transformer', m_t, 'transformer'), ('wire', m_w, 'wire')):
    r = m.val(data=DATA_YAML[key], split='test')
    print(f'{tag} TEST: mAP50-95={r.box.map:.3f}  mAP50={r.box.map50:.3f}')

In [ ]:
# Full cascade using THIS repo's inference code on a few test frames.
import glob, os, cv2
import matplotlib.pyplot as plt
from thermal.detector import YoloDetector
from thermal.colormap import build_lut, ColorToHeat
from thermal.pipeline import analyze_image
from thermal.report import annotate, to_json

td = YoloDetector('runs/transformer/yolo26x/weights/best.pt')
wd = YoloDetector('runs/wire/yolo26x/weights/best.pt')
c2h = ColorToHeat(build_lut('inferno'))

# run on the ORIG test frames (raw palette -> true heat)
imgs = sorted(glob.glob('/content/yolo_thermal_transformer/images/test/orig/*.jpg'))[:6]
os.makedirs('/content/cascade_out', exist_ok=True)
fig, axes = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4))
for ax, p in zip(axes if len(imgs) > 1 else [axes], imgs):
    bgr = cv2.imread(p); rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    findings, _, calib = analyze_image(rgb, td, wd, c2h)
    out = annotate(bgr, findings)
    cv2.imwrite(f"/content/cascade_out/{os.path.basename(p)}", out)
    print(os.path.basename(p), 'calib_ok=', calib, to_json(findings))
    ax.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout(); plt.show()

Annotated cascade outputs are in `/content/cascade_out/` and the weights in your Drive `thermal_weights/` folder. Download `transformer.pt` + `wire.pt` into the repo's `models/` to run the local API. If train-vs-val (`results.png`) diverges, try `MODEL='yolo26m.pt'`.